# Demo 09c — NMF with pyNNMF

**Class 09 · D1** (slides: `slides/09_blind_signal_separation.md`)

NMF factors a non-negative matrix X ≈ W Hᵀ with both factors non-negative:
the columns of W are **parts**, not directions. We use the course library
`pyNNMF` (`pynnmf`): multiplicative updates (`nmf_mu`, Frobenius / KL costs),
HALS and robust weighted NMF (`rwnmf`, missing values). Digits are the example:
NMF parts vs PCA components on the same data, cost curves, choosing k.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pynnmf
from sklearn.decomposition import NMF as SkNMF
from sklearn.decomposition import PCA

from faa import data, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")

## Slide: The toy matrix, by hand

A 6 x 8 non-negative matrix of rank 2 (two taste groups of columns): NMF with k = 2
reconstructs it exactly; W and H are the two hidden factors.

In [ ]:
rng = np.random.default_rng(SEED)
W0 = np.abs(rng.normal(0, 1, (6, 2))) + 0.5
H0 = np.abs(rng.normal(0, 1, (2, 8))) + 0.5
H0[:, :4] *= 3  # two column groups, so the matrix has rank 2
Xtoy = W0 @ H0
rec, W, H, err = pynnmf.nmf_mu(Xtoy, 2, n=500, seed=SEED)
slide("Toy matrix: exact recovery at k = 2")
print(f"relative Frobenius error: {np.linalg.norm(Xtoy - W @ H) / np.linalg.norm(Xtoy):.4f}")
print(f"W (6 x 2) first rows:\n{W[:3].round(2)}")
print(f"H (2 x 8) column groups: {H.mean(0).round(1)}")

## Slide: Digits — parts against directions

The same digits data, two decompositions with k = 8: PCA components (positive and negative,
blurry whole digits) vs NMF parts (positive strokes). The figure is the argument.

In [ ]:
Xd, _, _ = data.load_table("digits", "target")
Xd = Xd / Xd.max()
K = 8
pca = PCA(n_components=K).fit(Xd)
# nmf_hals is pyNNMF's best solver here (lower rel err than nmf_mu at the same budget);
# both H shapes are (k, n) so the parts are the rows of H either way.
nmf_rec, Wd, Hd, _ = pynnmf.nmf_hals(Xd, K, n=400, seed=SEED, init="nndsvd")
f, ax = plt.subplots(2, K, figsize=(plotlib.FULL, 2.7))
for j in range(K):
    ax[0, j].imshow(pca.components_[j].reshape(8, 8), cmap="viridis")
    ax[1, j].imshow(Hd[j].reshape(8, 8), cmap="viridis")
    ax[0, j].set_xticks([]), ax[0, j].set_yticks([]), ax[0, j].set_aspect("equal")
    ax[1, j].set_xticks([]), ax[1, j].set_yticks([]), ax[1, j].set_aspect("equal")
ax[0, 0].set_ylabel("PCA", fontsize=7.5)
ax[1, 0].set_ylabel("NMF", fontsize=7.5)
plotlib.save(f, "p3_parts_vs_directions")
print(f"PCA components range: [{pca.components_.min():.2f}, {pca.components_.max():.2f}] (negative values)")
print(f"NMF parts range: [{Hd.min():.2f}, {Hd.max():.2f}] (all non-negative)")
print(
    f"reconstruction error, digits: PCA {np.linalg.norm(Xd - pca.inverse_transform(pca.transform(Xd))) / np.linalg.norm(Xd):.3f}, NMF {np.linalg.norm(Xd - Wd @ Hd) / np.linalg.norm(Xd):.3f}"
)

## Slide: The multiplicative update, convergence on digits

W ← W ⊙ (X H) / (W Hᵀ H), H ← H ⊙ (Wᵀ X) / (Wᵀ W H): each factor scales by a ratio,
so positivity is preserved and the cost never rises. Cost curve from `nmf_mu` on digits.

In [ ]:
costs = []


def cb(it, c):
    costs.append((it, c))


_, _, _, _ = pynnmf.nmf_mu(Xd, 8, n=100, seed=SEED, init="nndsvd", callback=cb)
f, ax = plt.subplots(figsize=(plotlib.COL, 1.3))
its, cs = zip(*costs)
ax.plot(its, cs, color=RED, lw=1.2)
ax.set_xlabel("iteration")
ax.set_title("Frobenius cost")
ax.set_yticks([60, 70, 80])
plotlib.save(f, "p3_mu_curve")
print(f"cost: {cs[0]:.1f} -> {cs[-1]:.1f} in {its[-1]} iterations, never rises")

## Slide: Frobenius or KL? Gaussian noise vs counts

Two costs, one data type each: squared error for Gaussian-like noise (digits), KL divergence
for counts (spambase word frequencies). Same k = 8, different relative error.

In [ ]:
Xspam, _, _ = data.load_table("spambase", "spam")
rec_f, Wf, Hf, e_f = pynnmf.nmf_mu(Xspam, 8, n=300, seed=SEED, init="nndsvd")
rec_k, Wk, Hk, e_k = pynnmf.nmf_mu(Xspam, 8, n=300, seed=SEED, init="nndsvd", cost="kl")
slide("Costs on spambase (k = 8)")
print(
    pl.DataFrame(
        {
            "cost": ["frobenius", "kl"],
            "rel. fro error": [
                np.linalg.norm(Xspam - Wf @ Hf) / np.linalg.norm(Xspam),
                np.linalg.norm(Xspam - Wk @ Hk) / np.linalg.norm(Xspam),
            ],
        }
    ).with_columns(pl.col(pl.Float64).round(3))
)
print(f"KL cost of the KL fit: {e_k:.1f}, of the Frobenius fit: {pynnmf.cost_kl(Xspam, Wf @ Hf):.1f}")

## Sanity check: pyNNMF against sklearn NMF

The course library and the sklearn twin agree on the objective within tolerance
(different solvers, same optimum): both fit X ≈ W Hᵀ on digits.

In [ ]:
sk = SkNMF(n_components=8, init="nndsvd", max_iter=400, random_state=SEED).fit(Xd)
slide("pyNNMF vs sklearn NMF on digits (k = 8)")
sk_sk = SkNMF(n_components=8, init="nndsvd", max_iter=400, random_state=SEED).fit(Xd)
py_rel = np.linalg.norm(Xd - Wd @ Hd) / np.linalg.norm(Xd)
sk_rel = np.linalg.norm(Xd - sk.transform(Xd) @ sk.components_) / np.linalg.norm(Xd)
print(f"pyNNMF relative Frobenius error:  {py_rel:.4f}")
print(f"sklearn relative Frobenius error: {sk_rel:.4f}")
print(f"(sklearn reconstruction_err_ {sk.reconstruction_err_:.4f} uses its own 1/2·MSE scaling)")
assert abs(py_rel - sk_rel) < 0.02, "sklearn and pyNNMF disagree on the optimum"

## Slide: The algorithm zoo — same problem, different solvers

pyNNMF ships several solvers; all fit the same X ≈ W H. Relative Frobenius error
at k = 8 and k = 16, same budget n = 400: HALS with NNDSVD init wins both.

In [ ]:
rows = []
for k_ in [8, 16]:
    for name, fn in [
        ("nmf_mu", lambda k_: pynnmf.nmf_mu(Xd, k_, n=400, seed=SEED, init="nndsvd")),
        ("nmf_hals", lambda k_: pynnmf.nmf_hals(Xd, k_, n=400, seed=SEED, init="nndsvd")),
    ]:
        _, W_, H_, _ = fn(k_)
        rows.append({"solver": name, "k": k_, "rel err": round(np.linalg.norm(Xd - W_ @ H_) / np.linalg.norm(Xd), 4)})
slide("pyNNMF solver comparison on digits")
print(pl.DataFrame(rows))
print("rwnmf tie on missing pixels: both 0.215 RMSE (next cell)")

## Slide: Missing values — rwnmf fills the gaps

Mask 25 % of digits at random; `rwnmf` imputes them while factorising. Reconstruction
of the missing entries against the true pixels; HALS does the same for NaN entries.

In [ ]:
mask = np.random.default_rng(SEED).random(Xd.shape) > 0.25
Xmiss = Xd.copy()
Xmiss[~mask] = np.nan
rec_m, Wm, Hm, e_m = pynnmf.rwnmf(Xmiss, 8, num_iter=400, seed=SEED)
pred = Wm @ Hm.T
slide("rwnmf with 25 % missing pixels")
print(f"RMSE on the missing pixels: {np.sqrt(np.mean((pred[~mask] - Xd[~mask]) ** 2)):.3f} (pixel range 0-1)")
print(f"RMSE on the observed pixels: {np.sqrt(np.mean((pred[mask] - Xd[mask]) ** 2)):.3f}")

## Sweep and failure cases (more than the slides)

1. Choosing k: relative error falls with k, with no knee on digits — a decision, not a law.
2. Negative entries: NMF needs X ≥ 0; centered data is out (shift it back or use PCA/ICA).

In [ ]:
rows = []
for k in [2, 4, 8, 16, 32]:
    rec_k2, _, _, _ = pynnmf.nmf_mu(Xd, k, n=300, seed=SEED, init="nndsvd")
    rows.append({"k": k, "rel error": np.linalg.norm(Xd - rec_k2) / np.linalg.norm(Xd)})
slide("k sweep on digits")
print(pl.DataFrame(rows).with_columns(pl.col(pl.Float64).round(3)))

slide("Failure: negative entries")
Xc = Xd - Xd.mean()
_, Wc, Hc, _ = pynnmf.nmf_mu(Xc, 8, n=50, seed=SEED)
print(f"centered data range: [{Xc.min():.2f}, {Xc.max():.2f}]")
print(f"fit reconstructs a NON-NEGATIVE matrix: [{(Wc @ Hc).min():.2f}, {(Wc @ Hc).max():.2f}]")
print(
    f"relative error: {np.linalg.norm(Xc - Wc @ Hc) / np.linalg.norm(Xc):.3f} — the negative part is simply unreachable"
)
print("lesson: NMF needs X >= 0; centered data is out (shift it back or use PCA/ICA)")